In [16]:
from sentence_transformers import SentenceTransformer
import torch
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)

def create_embedding(texts):
    if type(texts) == list: # the input param is a list contains text chunks
        embeddings = model.encode(texts)
        return embeddings
    elif type(texts) == str: # just one text chunk
        embedding = model.encode([texts])
        return embedding
    else:
        raise ValueError("The texts paramters should be str or list[str]")

print(device)

cuda


In [17]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\testing
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [18]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response.text, response.usage_metadata.total_token_count

In [19]:
import sys
import numpy as np
sys.path.append(root_path)
from testing.metrics.faithfulness import compute_faithfulness #Detect hallucinations: Measures how much the LLM answer is based on the retrieved context
"""
question: str,
answer: str,
contexts: List[str],
call_gemini,
max_retries: int = 5
"""
from testing.metrics.context_recall import compute_context_recall #Evaluate retrieval coverage: Measures how much of the reference (ground-truth) answer is supported by the retrieved context.
"""
question: str,
contexts: List[str],
reference_answer: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.coverage import compute_coverage #Measures how completely a model’s response covers the factual content of a reference (ground-truth) answer.
"""
question: str,
reference: str,
response: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.rouge import compute_rouge
#Calculate F-1 of lexical overlap by LCS: longest common subsequence (in terms of word order, adjacency not required)
#Precision = len(LCS) / len(Answer)
#Recall = len(LCS) / len(Ref)
"""
answer: str,
ground_truth: str,
rouge_type: str = "rougeL",
mode: str = "fmeasure"
"""
from testing.metrics.accuracy import compute_answer_accuracy
"""
question: str,
answer: str,
ground_truth: str,
call_gemini,
create_embedding,
weights: List[float] = [0.75, 0.25],
beta: float = 1.0,
max_retries: int = 5
"""
print("Import evaluation functions")


Import evaluation functions


In [20]:
import pandas as pd
medical_questions_answered_loaded = pd.read_parquet("data/normal_rag_answer.parquet")
relevant_questions = medical_questions_answered_loaded[medical_questions_answered_loaded["question_type"] == "Complex Reasoning"]
question_ids = relevant_questions['id'].to_list()
relevant_questions

,id,source,question,answer,question_type,evidence,evidence_relations,LLM_answer,LLM_context,LLM_tokens
0,Medical-604c9d44,Medical,Why is a patient with fair skin and a history ...,Because both fair skin and immune suppression ...,Complex Reasoning,Fair skin is an independent risk factor for ba...,"Fair skin, light hair, and light eye color inc...",A patient with fair skin is at a higher risk f...,Here are distinct categories of high-level inf...,2727.0
1,Medical-df366063,Medical,If a patient presents with a shiny bump on the...,"A medical and family history, physical and ski...",Complex Reasoning,A medical and family history should be perform...,BCC presents as shiny bumps; BCC most commonly...,None,None,NaN
2,Medical-0a90f1f2,Medical,How does the management of recurrent basal cel...,Management of recurrence depends on risk and r...,Complex Reasoning,Management of recurrent basal cell carcinoma d...,Recurrence refers to BCC returning to the orig...,None,None,NaN
3,Medical-831bb1e6,Medical,Which combination of risk factors should promp...,"A combination of older age, fair skin, and fam...",Complex Reasoning,Older age is a risk factor for basal cell carc...,Older age is associated with higher risk of BC...,None,None,NaN
4,Medical-01e3a9ea,Medical,Why might a patient with a shiny bump and a hi...,Because radiation therapy is a risk factor for...,Complex Reasoning,A shiny bump can be a symptom of basal cell ca...,Radiation therapy is a risk factor for BCC; BC...,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...
504,Medical-1eda4787,Medical,Which patients with breast cancer should recei...,All patients of reproductive potential should ...,Complex Reasoning,All patients of reproductive potential should ...,Fertility preservation should be discussed bef...,None,None,NaN
505,Medical-24f2e8ea,Medical,How does the presence of HER2 positivity alter...,HER2 positivity adds HER2-targeted therapy to ...,Complex Reasoning,HER2 positivity adds HER2-targeted therapy to ...,HER2-targeted therapy is used for HER2-positiv...,None,None,NaN
506,Medical-d1a87cb4,Medical,What combination of clinical findings and diag...,Symptoms like bone pain or neurological sympto...,Complex Reasoning,Symptoms like bone pain or neurological sympto...,Metastatic breast cancer (MBC) is breast cance...,None,None,NaN
507,Medical-311d66d6,Medical,Which risk factors should be evaluated in a pa...,"Family history, BRCA1/2 mutations, and assigne...",Complex Reasoning,Family history should be evaluated as a risk f...,"Risk factors include family history, BRCA muta...",None,None,NaN


In [21]:
try:
    medical_questions_evaluated = pd.read_parquet("data/evaluations/normal_rag_evaluated.parquet")
    medical_questions_evaluated = (
        medical_questions_evaluated
        .set_index("qid")
        .to_dict(orient="index")
    )
except:
    medical_questions_evaluated = {}


medical_questions_evaluated

{}

In [22]:
def all_valid(values):
    return all(
        v is not None and not np.isnan(v)
        for v in values
    )

In [ ]:
#accuracy
#rouge-L
#context recall

from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 10
SLEEP_SEC = 30

def get_answers():
    for qid in tqdm(question_ids):
        if qid in medical_questions_evaluated:
            continue

        row = relevant_questions.loc[relevant_questions["id"] == qid].iloc[0]
        row_eval = {}

        question = row["question"]
        reference = row["answer"]
        answer = row["LLM_answer"]
        context = row["LLM_context"]
        contexts = context.split(sep)
        
        rouge_score = compute_rouge(answer, reference)
        accuracy, token1 = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                accuracy, token1 = compute_answer_accuracy(question, answer, reference, call_gemini, create_embedding)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} accuracy calculation failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff

        context_recall, token2 = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                context_recall, token2 = compute_context_recall(question, contexts, reference, call_gemini)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} context recall calculation failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff
        if all_valid([accuracy, context_recall, token1, token2]):
            row_eval["accuracy"] = accuracy
            row_eval["rouge"] = rouge_score
            row_eval["context_recall"] = context_recall
            row_eval["tokens"] = token1 + token2
            medical_questions_evaluated[qid] = row_eval
get_answers()


  0%|          | 0/509 [00:28<?, ?it/s]


In [24]:
medical_questions_evaluated

{'Medical-604c9d44': {'accuracy': 0.6501265572160295,
  'rouge': 0.16129032258064516,
  'context_recall': 1.0,
  'tokens': 8407}}

In [25]:
df_eval = (
    pd.DataFrame.from_dict(medical_questions_evaluated, orient="index")
      .reset_index()
      .rename(columns={"index": "qid"})
)

In [26]:
df_eval.to_parquet("data/evaluations/normal_rag_evaluated.parquet")
df_eval_loaded = pd.read_parquet("data/evaluations/normal_rag_evaluated.parquet")
df_eval_loaded

,qid,accuracy,rouge,context_recall,tokens
0,Medical-604c9d44,0.650127,0.16129,1.0,8407


In [27]:
print("min:", df_eval_loaded["tokens"].min())
print("max:", df_eval_loaded["tokens"].max())
print("avg:", df_eval_loaded["tokens"].mean())
print("sum:", df_eval_loaded["tokens"].sum())

min: 8407
max: 8407
avg: 8407.0
sum: 8407


In [28]:
print("min:", df_eval_loaded["accuracy"].min())
print("max:", df_eval_loaded["accuracy"].max())
print("avg:", df_eval_loaded["accuracy"].mean())
print("sum:", df_eval_loaded["accuracy"].sum())

min: 0.6501265572160295
max: 0.6501265572160295
avg: 0.6501265572160295
sum: 0.6501265572160295


In [29]:
print("min:", df_eval_loaded["rouge"].min())
print("max:", df_eval_loaded["rouge"].max())
print("avg:", df_eval_loaded["rouge"].mean())
print("sum:", df_eval_loaded["rouge"].sum())

min: 0.16129032258064516
max: 0.16129032258064516
avg: 0.16129032258064516
sum: 0.16129032258064516


In [30]:
print("min:", df_eval_loaded["context_recall"].min())
print("max:", df_eval_loaded["context_recall"].max())
print("avg:", df_eval_loaded["context_recall"].mean())
print("sum:", df_eval_loaded["context_recall"].sum())

min: 1.0
max: 1.0
avg: 1.0
sum: 1.0
